In [1]:
%pip install liac-arff


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import arff
import pandas as pd

with open("dataset.arff", "r", encoding="utf-8") as file:
    dataset = arff.load(file)

df = pd.DataFrame(
    dataset["data"],
    columns=[attr[0] for attr in dataset["attributes"]]
)

# Missing value
df = df.replace("?", pd.NA)

print("Dataset loaded successfully!")
print("Rows and columns:", df.shape)
display(df.head())

Dataset loaded successfully!
Rows and columns: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("All libraries imported successfully!")

All libraries imported successfully!


In [4]:
print("Dataset shape:", df.shape)
print("SalePrice column exists:", "SalePrice" in df.columns)
print("Target data type:", df["SalePrice"].dtype)

display(df[["SalePrice"]].head())

Dataset shape: (1460, 81)
SalePrice column exists: True
Target data type: int64


,SalePrice
0,208500
1,181500
2,223500
3,140000
4,250000


In [5]:
# Separate input features and target
X = df.drop(columns=["SalePrice", "Id"], errors="ignore")
y = df["SalePrice"]

# Split before preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (1168, 79)
Testing features: (292, 79)
Training target: (1168,)
Testing target: (292,)


In [9]:
# Identify numerical columns
numerical_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

# Identify categorical columns
categorical_cols = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Number of numerical columns:", len(numerical_cols))
print("Number of categorical columns:", len(categorical_cols))

print("\nFirst 10 numerical columns:")
print(numerical_cols[:10])

print("\nFirst 10 categorical columns:")
print(categorical_cols[:10])

Number of numerical columns: 36
Number of categorical columns: 43

First 10 numerical columns:
['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2']

First 10 categorical columns:
['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1']


In [6]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical pipeline created!")

Numerical pipeline created!


In [7]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

print("Categorical pipeline created!")

Categorical pipeline created!


In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ]
)

print("ColumnTransformer created!")

ColumnTransformer created!


In [11]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training data after preprocessing:", X_train_processed.shape)
print("Testing data after preprocessing:", X_test_processed.shape)

Training data after preprocessing: (1168, 286)
Testing data after preprocessing: (292, 286)


In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Combine preprocessing and model into one pipeline
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Train using training data only
model_pipeline.fit(X_train, y_train)

# Predict on unseen test data
y_pred = model_pipeline.predict(X_test)

print("Model training completed!")
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

Model training completed!
Mean Absolute Error: 17750.58373287671
R2 Score: 0.8876872398529598


In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline

models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}

model_pipelines = {}

for name, model in models.items():
    model_pipelines[name] = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

print("All 4 model pipelines created successfully!")

All 4 model pipelines created successfully!


In [14]:
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for name, pipeline in model_pipelines.items():

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    mae_scores = -scores

    cv_results.append({
        "Model": name,
        "Mean MAE": mae_scores.mean(),
        "Std MAE": mae_scores.std()
    })

cv_df = pd.DataFrame(cv_results)

cv_df = cv_df.sort_values(
    by="Mean MAE"
)

display(cv_df)

,Model,Mean MAE,Std MAE
3,Gradient Boosting,16058.834722,1424.591341
2,Random Forest,18270.135956,1154.775605
0,Linear Regression,19728.321018,783.168832
1,Decision Tree,26584.787403,2020.110462


In [15]:
from sklearn.model_selection import GridSearchCV

gb_pipeline = model_pipelines["Gradient Boosting"]

param_grid = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3, 4]
}

grid_search = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation MAE:")
print(-grid_search.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Parameters:
{'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}

Best Cross-Validation MAE:
15609.620413330942


In [22]:
import numpy as np
from sklearn.metrics import mean_squared_error

In [23]:
best_model = grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_test)

print("Tuned Model Test Results")
print("MAE:", mean_absolute_error(y_test, y_pred_tuned))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_tuned)))
print("R2 Score:", r2_score(y_test, y_pred_tuned))

Tuned Model Test Results
MAE: 16357.558309294309
RMSE: 27245.390042685078
R2 Score: 0.9032229844563202


In [25]:
comparison_results = []

for name, pipeline in model_pipelines.items():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    comparison_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2 Score": r2_score(y_test, predictions)
    })

# Add tuned Gradient Boosting
comparison_results.append({
    "Model": "Tuned Gradient Boosting",
    "MAE": mean_absolute_error(y_test, y_pred_tuned),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_tuned)),
    "R2 Score": r2_score(y_test, y_pred_tuned)
})

comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values(by="MAE")

display(comparison_df)

,Model,MAE,RMSE,R2 Score
4,Tuned Gradient Boosting,16357.558309,27245.390043,0.903223
3,Gradient Boosting,16824.920629,27725.658741,0.899781
2,Random Forest,17750.583733,29350.903011,0.887687
0,Linear Regression,18362.243869,29445.756981,0.886960
1,Decision Tree,28882.773973,44841.581134,0.737851


In [26]:
import joblib

joblib.dump(best_model, "best_house_price_model.joblib")

print("Best model saved successfully!")

Best model saved successfully!
